In [0]:
# 1. Define the parameters (Default value, Widget Name, UI Label)
dbutils.widgets.text("year", "", "Year to process") # There are various widgets to choose from like Multiselect,etc
# 2. Get the string value from the workflow
year = dbutils.widgets.get("year")


In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install ../databricks_helpers

In [0]:
from databricks_helpers.databricks_helper import DatabricksHelpers
from pyspark.sql.functions import col
import os
import pyspark.pandas as pd

dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

bronze_weather_data_table="moyalanjintos_catalog.aviation_analytics.bronze_weather_data"

weather_streaming_dir= f"{dbh.volume_directory()}/weather_stream"
os.makedirs(weather_streaming_dir, exist_ok=True)

schemas_dir=f"{dbh.schemas_directory()}/weather_bronze"
os.makedirs(schemas_dir, exist_ok=True)
checkpoints_dir=f"{dbh.checkpoints_directory()}/weather_bronze"
os.makedirs(checkpoints_dir, exist_ok=True)

if year :
    dbutils.fs.cp(f"{dbh.volume_directory()}/meteo_data/meteo_weather_data_{year}/", f"{weather_streaming_dir}/meteo_weather_data_f{year}", recurse=True)

### Creating bronze layer for Weather data

With schema evolution

In [0]:
from pyspark.sql.functions import col


# 1. Configure the Read Stream (Handling JSON (semi structured) & Schema Evolution)
weather_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schemas_dir)
    # JSON requires inference so numbers aren't treated as strings
    .option("cloudFiles.inferColumnTypes", "true") 
    # Tell Auto Loader to dynamically track new columns, helps in schema evolution
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    # Quarantine any corrupted JSON strings
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(weather_streaming_dir)
    # Extract metadata for perfect idempotency and data lineage
    .select(
        "*",
        col("_metadata.file_modification_time").alias("file_arrival_timestamp"),
        col("_metadata.file_path").alias("source_file_path")
    )
)

# 2. Configure the Write Stream (to a Delta Table)
(weather_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoints_dir)
    
    # Tell the Delta Table it is allowed to expand its schema
    .option("mergeSchema", "true") 
    
    .trigger(availableNow=True)
    .table(bronze_weather_data_table)
)